# Nemotron-3.5 Bolt E4B — device-assistant tool calling on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/nemotron-3.5-bolt-e4b-tool-agent.ipynb)

[`nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16) is NVIDIA's small Bolt checkpoint: Gemma-4-E4B further post-trained for multi-turn tool use, instruction following, and input grounding. The intended jobs are local voice/device assistants, interactive personas, and embedded interfaces — not a general 8B chat dump. The Hub file is BF16 and **15.9 GB**. That is the full weight file, not a measured Colab peak.

**Overlap:** this is the same Gemma 4 E4B family as [`gemma4_e4b_decision_email_triage.ipynb`](gemma4_e4b_decision_email_triage.ipynb). That notebook trains a decision head for support-email routing. This one does **not** train anything. It runs the NVIDIA Bolt checkpoint as a multi-turn **device assistant** with tools.

**Licence:** [OpenMDW-1.1](https://openmdw.ai/license/1-1/). The Gemma 4 base is Apache 2.0.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A T4 has about 15 GB of usable VRAM, no native bfloat16, and no FlashAttention 2. BF16 will not leave a working margin. This notebook loads the official BF16 weights in **4-bit NF4** with **bitsandbytes** and **SDPA**. The cells print `torch` allocated / reserved / peak memory and `nvidia-smi` after load and after generation — those are the numbers to trust, not an estimate written here.

Official Q4_K_M GGUF lives at [`nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB) (about 6.2 GB on the Hub). Its deployment line still says **llama.cpp (recipe pending)**, so this notebook stays on transformers + 4-bit.

The packaged `chat_template.jinja` is a Nemotron-updated Gemma 4 template. Tools are OpenAI-style function schemas. The model emits Gemma 4 calls (`<|tool_call>call:name{...}<tool_call|>`). Thinking is a template flag (`enable_thinking=True/False`); the template default is off. Recommended sampling on the card is temperature 1.0 and top_p 0.95. These demos use **greedy** decode and **modest** `max_new_tokens` so a free T4 can finish.

**Links:** [BF16 model card](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16) · [4GB GGUF](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB) · [blog](https://research.nvidia.com/labs/nemotron/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB/) · [technical report](https://research.nvidia.com/labs/nemotron/files/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB-Technical-Report.pdf) · [OpenMDW-1.1](https://github.com/OpenMDW/OpenMDW/blob/main/1.1/LICENSE.OpenMDW-1.1)


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate>=1.11.0" "bitsandbytes>=0.48.0" pillow

# The BF16 checkpoint was saved with transformers 5.5.0.dev0 (Gemma4ForConditionalGeneration).
# 5.20.1 matches the other recent T4 notebooks in this repo and is above that floor.
# bitsandbytes does the 4-bit load. accelerate places the quantized weights on the GPU.
# pillow is listed because the class is image-text-to-text; this notebook never sends an image.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.


In [ ]:
import subprocess

import torch
import transformers
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.5.0"), (
    f"transformers {transformers.__version__} is loaded; Bolt-E4B needs a Gemma 4 build (5.5+). "
    "Run the install cell, then Runtime > Restart session."
)

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on the mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
COMPUTE_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| 4-bit compute dtype:", COMPUTE_DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(subprocess.check_output(["nvidia-smi"], text=True))


## 1. Load Bolt-E4B in 4-bit

`AutoProcessor` + `AutoModelForImageTextToText`, the same pair as the checkpoint's
[`handler.py`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16/blob/main/handler.py).
That loads the packaged `chat_template.jinja` (Nemotron's Gemma 4 tool / thinking template).
The architecture is `Gemma4ForConditionalGeneration`. The card is text-in / text-out; vision
and audio stay unused.

The first run downloads the **15.9 GB** BF16 shard and quantizes it to NF4 on the fly. Colab
disk is large enough. After load, the next cell prints real allocated VRAM — do not treat the
weight-file size as the peak.


In [ ]:
import subprocess

from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16"


def print_vram(label):
    # Measured torch CUDA stats plus nvidia-smi used/total. Not an estimate.
    torch.cuda.synchronize()
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    print(
        f"{label}: torch allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | "
        f"peak allocated {peak:.2f} GB"
    )
    print(f"{label}: nvidia-smi used/total {smi}")


bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# Official handler uses AutoModelForImageTextToText + the repo chat template.
# 4-bit is the T4 path; BF16 is the 15.9 GB Hub file and does not leave headroom on 16 GB.
try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map="auto",
        attn_implementation="sdpa",
        trust_remote_code=True,
    )
    print("attention: sdpa")
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb,
        device_map="auto",
        attn_implementation="eager",
        trust_remote_code=True,
    )
    print("attention: eager")

model.eval()
print("parameters (reported): 8B total / 4B effective")
print_vram("after load")


## Helpers

`generate_reply` mirrors the official handler: `apply_chat_template(..., tools=..., enable_thinking=...)`,
then `processor(text=[prompt])`. Decode with `skip_special_tokens=False` so Gemma 4 channel and
tool-call delimiters stay intact.

Greedy decode keeps the demos cheap on a T4. The card's recommended sampler is temperature 1.0,
top_p 0.95; switch `do_sample=True` if you want that. Keep `max_new_tokens` modest — thinking
and tool traces share the same budget.


In [ ]:
import json
import re

DEVICE_SYSTEM = (
    "You are a local device assistant on a phone. "
    "Use tools when you need live data or to change device state. "
    "Do not invent weather, reminders, or notes. "
    "After tools finish, answer in one or two short sentences."
)


def generate_reply(
    messages,
    tools=None,
    enable_thinking=False,
    max_new_tokens=160,
    conversational_mode=False,
):
    prompt = processor.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
        conversational_mode=conversational_mode,
    )
    model_inputs = processor(text=[prompt], return_tensors="pt")
    tokenizer = getattr(processor, "tokenizer", processor)
    model_device = next(p.device for p in model.parameters() if p.device.type != "meta")
    model_inputs = {key: value.to(model_device) for key, value in model_inputs.items()}
    input_len = model_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = generated[0, input_len:]
    text = processor.decode(
        new_tokens,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    for token in (tokenizer.eos_token, tokenizer.pad_token, "<turn|>"):
        if token:
            while text.endswith(token):
                text = text[: -len(token)]
    print_vram(f"after generate ({new_tokens.shape[0]} new tokens)")
    return text.strip()


def split_thought(text):
    # Pull Gemma 4 thought channels out so the printed answer is readable.
    thoughts = re.findall(
        r"<\|channel\|?>thought\n?(.*?)<channel\|>",
        text,
        flags=re.DOTALL,
    )
    answer = re.sub(
        r"<\|channel\|?>thought\n?.*?<channel\|>",
        "",
        text,
        flags=re.DOTALL,
    )
    answer = re.sub(r"<\|/?think\|?>", "", answer)
    answer = re.sub(r"<\|turn\|?>\w*\n?", "", answer)
    return [item.strip() for item in thoughts if item.strip()], answer.strip()


def _parse_gemma_value(src, i=0):
    # Parse one Gemma 4 tool-argument value starting at src[i].
    delim = '<|"|>'
    n = len(src)
    while i < n and src[i] in " \n\t":
        i += 1
    if src.startswith(delim, i):
        i += len(delim)
        end = src.find(delim, i)
        if end < 0:
            raise ValueError("unterminated Gemma string")
        return src[i:end], end + len(delim)
    if src.startswith("true", i):
        return True, i + 4
    if src.startswith("false", i):
        return False, i + 5
    if src.startswith("null", i):
        return None, i + 4
    if src.startswith("{", i):
        obj, i = {}, i + 1
        while i < n:
            while i < n and src[i] in " \n\t,":
                i += 1
            if i < n and src[i] == "}":
                return obj, i + 1
            key, i = _parse_gemma_key(src, i)
            if i < n and src[i] == ":":
                i += 1
            value, i = _parse_gemma_value(src, i)
            obj[key] = value
        raise ValueError("unterminated Gemma object")
    if src.startswith("[", i):
        arr, i = [], i + 1
        while i < n:
            while i < n and src[i] in " \n\t,":
                i += 1
            if i < n and src[i] == "]":
                return arr, i + 1
            value, i = _parse_gemma_value(src, i)
            arr.append(value)
        raise ValueError("unterminated Gemma array")
    match = re.match(r"-?\d+(?:\.\d+)?", src[i:])
    if match:
        raw = match.group(0)
        number = float(raw) if "." in raw else int(raw)
        return number, i + len(raw)
    raise ValueError(f"cannot parse Gemma value at {src[i:i+40]!r}")


def _parse_gemma_key(src, i):
    delim = '<|"|>'
    while i < len(src) and src[i] in " \n\t":
        i += 1
    if src.startswith(delim, i):
        key, i = _parse_gemma_value(src, i)
        return key, i
    match = re.match(r"[A-Za-z0-9_]+", src[i:])
    if not match:
        raise ValueError(f"cannot parse Gemma key at {src[i:i+40]!r}")
    return match.group(0), i + len(match.group(0))


def parse_tool_calls(text):
    # Read <|tool_call>call:name{args}<tool_call|> blocks from a completion.
    pattern = re.compile(
        r"<\|tool_call\|?>call:([A-Za-z0-9_]+)\{(.*?)\}<tool_call\|>",
        flags=re.DOTALL,
    )
    calls = []
    for index, match in enumerate(pattern.finditer(text), start=1):
        name, raw_args = match.group(1), match.group(2).strip()
        args = {}
        if raw_args:
            parsed, _ = _parse_gemma_value("{" + raw_args + "}")
            args = parsed
        calls.append(
            {
                "id": f"call_{index}",
                "type": "function",
                "function": {"name": name, "arguments": args},
            }
        )
    return calls


_SAMPLE_CALL = '<|tool_call>call:get_weather{city:<|"|>Santa Clara<|"|>}<tool_call|>'
_parsed_sample = parse_tool_calls(_SAMPLE_CALL)
assert _parsed_sample[0]["function"]["name"] == "get_weather"
assert _parsed_sample[0]["function"]["arguments"]["city"] == "Santa Clara"
print("tool parser ok:", _parsed_sample[0]["function"])


## 2. Demo 1 — chat, thinking on and off

The packaged template accepts `enable_thinking`. Off is the template default and is the
right setting for a spoken device reply. On writes a thought channel before the answer.
Both runs use the same short question so you can see the extra tokens thinking spends.


In [ ]:
question = "In one sentence, what should a device assistant do when it does not know an answer?"
base_messages = [
    {"role": "system", "content": DEVICE_SYSTEM},
    {"role": "user", "content": question},
]

print("=== thinking OFF (template default) ===")
off_text = generate_reply(base_messages, enable_thinking=False, max_new_tokens=80)
_, off_answer = split_thought(off_text)
print(off_answer or off_text)

print("\n=== thinking ON ===")
on_text = generate_reply(base_messages, enable_thinking=True, max_new_tokens=192)
thoughts, on_answer = split_thought(on_text)
if thoughts:
    print("[thought]")
    print(thoughts[0][:800])
    print()
print("[answer]")
print(on_answer or on_text)


## 3. Demo 2 — multi-turn tool-calling agent

Three mock device tools: `get_weather`, `set_reminder`, `search_notes`. Schemas are the
OpenAI function objects the template expects. Completions are parsed with the Gemma 4
call format from `chat_template.jinja`:

```
<|tool_call>call:get_weather{city:<|"|>Santa Clara<|"|>}<tool_call|>
```

When the loop feeds a tool result back, `function.arguments` must be a **mapping**, not a
JSON string — the template raises if you pass a string. A `tool` message has to follow the
assistant turn that produced `tool_calls` immediately (the card's stray-tool-response rule).

The user asks two follow-ups in one conversation so you see weather → reminder → notes,
not three isolated one-shots.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city. Use this instead of guessing.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, for example Santa Clara"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_reminder",
            "description": "Create a local reminder on the device.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Reminder body"},
                    "when": {"type": "string", "description": "When to fire, as the user said it"},
                },
                "required": ["text", "when"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_notes",
            "description": "Search the on-device notes store by keyword.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Keywords to match"},
                },
                "required": ["query"],
            },
        },
    },
]

REMINDERS = []
NOTES = [
    {"title": "Sunday market", "body": "Farmer's market Sunday 9am. Bring a tote and cash."},
    {"title": "Hall closet", "body": "Umbrella and rain jacket are in the hall closet."},
    {"title": "Sam dinner", "body": "Dinner with Sam at home tonight at 19:00. Accepted."},
]


def get_weather(city):
    catalog = {
        "santa clara": {
            "city": "Santa Clara",
            "temp_c": 16,
            "condition": "light rain",
            "precip_chance": 0.7,
        },
        "manila": {
            "city": "Manila",
            "temp_c": 29,
            "condition": "humid, partly cloudy",
            "precip_chance": 0.2,
        },
    }
    return catalog.get(
        city.strip().lower(),
        {"city": city, "temp_c": None, "condition": "unknown city in the mock catalog"},
    )


def set_reminder(text, when):
    item = {"id": f"rem_{len(REMINDERS) + 1}", "text": text, "when": when, "status": "scheduled"}
    REMINDERS.append(item)
    return item


def search_notes(query):
    needles = query.lower().split()
    hits = [
        note
        for note in NOTES
        if any(word in (note["title"] + " " + note["body"]).lower() for word in needles)
    ]
    return {"query": query, "matches": hits}


IMPLEMENTATIONS = {
    "get_weather": lambda args: get_weather(args.get("city", "")),
    "set_reminder": lambda args: set_reminder(args.get("text", ""), args.get("when", "")),
    "search_notes": lambda args: search_notes(args.get("query", "")),
}


def run_agent(user_turns, max_rounds=8):
    messages = [
        {"role": "system", "content": DEVICE_SYSTEM},
        {"role": "user", "content": user_turns[0]},
    ]
    pending_users = list(user_turns[1:])

    for step in range(1, max_rounds + 1):
        print(f"\n----- agent step {step} -----")
        raw = generate_reply(
            messages,
            tools=TOOLS,
            enable_thinking=False,
            max_new_tokens=192,
        )  # thinking off so a 192-token cap covers a call + a short reply
        calls = parse_tool_calls(raw)
        _, spoken = split_thought(raw)
        spoken_only = re.sub(
            r"<\|tool_call\|?>.*?<tool_call\|>",
            "",
            spoken,
            flags=re.DOTALL,
        ).strip()

        if not calls:
            print("model (no tool):")
            print(spoken_only or raw)
            if not pending_users:
                return
            nxt = pending_users.pop(0)
            print(f"\nuser: {nxt}")
            messages.append({"role": "assistant", "content": spoken_only or raw})
            messages.append({"role": "user", "content": nxt})
            continue

        print("model tool calls:")
        print(json.dumps(calls, indent=2))
        if spoken_only:
            print("model text before tools:", spoken_only)

        messages.append(
            {
                "role": "assistant",
                "content": spoken_only,
                "tool_calls": calls,
            }
        )
        for call in calls:
            name = call["function"]["name"]
            args = call["function"]["arguments"]
            impl = IMPLEMENTATIONS.get(name)
            result = impl(args) if impl else {"error": f"unknown tool {name}"}
            print(f"tool {name}({args}) -> {result}")
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call["id"],
                    "name": name,
                    "content": json.dumps(result),
                }
            )

    print("stopped: max_rounds reached")


run_agent(
    [
        "What's the weather in Santa Clara right now?",
        "If rain is likely, set a 7am reminder to pack the umbrella.",
        "Search my notes for umbrella or rain.",
    ]
)
print("\nreminders now:", REMINDERS)


## 4. Demo 3 — grounding and instruction following

Bolt is post-trained to stay on the provided state and to follow tight reply rules. That is
the difference that matters for a device assistant: do not invent a calendar entry, a note,
or the weather when the snapshot does not contain it.

The next cell gives a small on-device state and three questions. No tools. The model should
cite only what is in the snapshot, refuse the weather question, and keep the required shape.


In [ ]:
DEVICE_STATE = '''# CURRENT STATE
location: kitchen
local_time: 2026-10-26 18:42
battery_percent: 37
calendar:
- 19:00 "Dinner with Sam at home" (accepted)
notes:
- "Farmer's market Sunday 9am, bring tote and cash"
- "Umbrella is in the hall closet"
'''

GROUNDING_SYSTEM = (
    DEVICE_SYSTEM
    + " Ground every answer in CURRENT STATE. If the state does not contain the fact, "
    "say you do not know. Do not use outside knowledge. "
    "Reply with exactly two lines and nothing else:\n"
    "FACT: <one sentence>\n"
    "SOURCE: <state field or NONE>"
)

prompts = [
    "When is dinner and who is it with?",
    "What is the weather outside right now?",
    "Where is the umbrella? Keep the required two-line shape.",
]

for prompt in prompts:
    print("=" * 60)
    print("user:", prompt)
    text = generate_reply(
        [
            {"role": "system", "content": GROUNDING_SYSTEM + "\n\n" + DEVICE_STATE},
            {"role": "user", "content": prompt},
        ],
        enable_thinking=False,
        max_new_tokens=96,
    )
    _, answer = split_thought(text)
    print(answer or text)


## 5. Caveats

- **T4 VRAM.** The BF16 shard is 15.9 GB. A free T4 has about 15 GB usable. This notebook
  is the 4-bit path on purpose. If a cell OOMs, lower `max_new_tokens` or turn thinking off
  before you reach for a larger GPU. There is no A100-only path here.
- **Print the peak.** `print_vram` reports `torch` allocated / reserved / peak allocated and
  `nvidia-smi` used/total. Those are measured on *this* runtime. Weight-file size is only a
  lower bound.
- **Licence.** Weights are [OpenMDW-1.1](https://openmdw.ai/license/1-1/). Gemma 4 underneath
  is Apache 2.0. Read both before you ship anything.
- **Thinking vs tools.** Thinking and tool traces share `max_new_tokens`. The agent loop
  keeps thinking off so a 192-token cap is enough for a call plus a short reply.
- **Tool message order.** A `tool` role turn must follow the assistant turn that produced
  `tool_calls`. Do not insert a user/system message between them. Arguments fed back into
  the template must be a dict, not a JSON string.
- **GGUF.** [`NVIDIA-Nemotron-3.5-Bolt-E4B-4GB`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB)
  is the official Q4_K_M file. The card still marks the llama.cpp recipe as pending, which
  is why this notebook does not switch to GGUF.
- **Not the triage notebook.** [`gemma4_e4b_decision_email_triage.ipynb`](gemma4_e4b_decision_email_triage.ipynb)
  is the Gemma 4 E4B Unsloth decision-model trainer. Different job, different checkpoint.

**Links again:** [model card](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-BF16) ·
[4GB GGUF](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB) ·
[blog](https://research.nvidia.com/labs/nemotron/NVIDIA-Nemotron-3.5-Bolt-E4B-4GB/)
